# W14 · Project showcase / 專題成果展

**English.** Execute the extension you scoped in W13, produce the artifacts, and
self-assess against the rubric. Keep it **honest** — a negative result, reported
cleanly with a matched baseline, scores well. Historical course CSVs remain
legacy-unverified and are not evidence for your submission; Table 3's tracked
0.269–0.345 SDF IoUs are an honesty example, not a baseline to copy. Full rubric:
`docs/06_capstone.md`.

**繁體中文.** 執行 W13 界定的延伸,產出交付物,並對照評分表自評。保持**誠實**——
負結果只要有相同條件的基線且乾淨呈現,一樣拿分。既有課程 CSV 仍是
legacy-unverified,不能當成本次提交證據;Table 3 的 0.269–0.345 SDF IoU
只是誠實呈現範例,不能直接抄成基線。
完整評分表見 `docs/06_capstone.md`。

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import math, torch, matplotlib.pyplot as plt
from peps.train import auto_device
def _required_text(value, name):
    if not isinstance(value, str) or not value.strip() or 'TODO' in value:
        raise ValueError(f'{name} must be nonblank and contain no TODO')
    return value.strip()
device = auto_device(); print('device', device)

In [ ]:
# The four capstone tracks (see docs/06_capstone.md for the full brief + rubric).
TRACKS = {
  'a': {'name': 'Short 3D video volume (x, y, t)',
        'start': ['peps/wrapper.py', 'peps/projector.py', 'peps/encoders/grid.py',
                  'apps/image/ (as a template)', 'peps/train.py'],
        'deliverable': '3D grid vs 3D Grid-PEPS on one small licensed clip',
        'csv': 'results/capstone_video3d.csv'},
  'b': {'name': 'Quantization calibration or short QAT recovery',
        'start': ['peps/quant/ptq.py', 'notebooks/W10_quantization.ipynb',
                  'tests/test_quantization.py'],
        'deliverable': 'clipping/calibration or QAT vs rerun per-channel PTQ',
        'csv': 'results/capstone_quant_calibration.csv'},
  'c': {'name': 'Design + evaluate a new aggregator (beyond concat/pink/brownian)',
        'start': ['peps/aggregate.py', 'apps/image/build.py',
                  'notebooks/W06_pink_peps.ipynb'],
        'deliverable': 'new aggregator kind + params-vs-PSNR vs the existing three',
        'csv': 'results/capstone_aggregator.csv'},
  'd': {'name': 'End-to-end PEPS runtime optimization and receipt',
        'start': ['hip/wmma_mlp.hip', 'hip/fused_peps_kernel.hip',
                  'hip/bench_latency.sh', 'tests/test_hip_parity.py'],
        'deliverable': 'one optimization vs rerun full-pipeline baseline; parity + latency',
        'csv': 'results/capstone_runtime.csv'},
}

# >>> Pick your track here <<<
TRACK = 'a'  # TODO(student): one of 'a' | 'b' | 'c' | 'd'
t = TRACKS[TRACK]
print('Track', TRACK, '-', t['name'])
print('Starting files:'); [print('  -', s) for s in t['start']]
print('Deliverable   :', t['deliverable'])
print('Results CSV   :', t['csv'])

## 1. Load your W13 baseline / 載入 W13 基線
Pull back the baseline number you recorded last week so the comparison is
matched. 讀回上週記錄的基線,確保對照條件一致。

In [ ]:
import csv
base_path = f'../results/capstone_{TRACK}_baseline.csv'
if not os.path.exists(base_path):
    raise FileNotFoundError('Run W13 and record a verified baseline first')
with open(base_path) as f: baseline_rows = list(csv.DictReader(f))
if len(baseline_rows) != 1 or baseline_rows[0].get('status') != 'verified':
    raise ValueError('W13 baseline must contain exactly one verified row')
baseline_row = baseline_rows[0]
baseline_value = float(baseline_row['value'])
if not math.isfinite(baseline_value): raise ValueError('baseline must be finite')
print('baseline:', baseline_row)

## 2. Run your extension / 跑出延伸
The single change your project is about. Sketches per track below — fill the one
for your `TRACK`.

你的專題所在的那**一個**改動。以下為各軌道草圖,填入你 `TRACK` 對應者。

In [ ]:
# TODO(student): implement your extension for the chosen track.
if TRACK == 'a':
    # Supported scope: a short (x,y,t) volume through GridEncoder(dim=3).
    print('a) fit 3D Grid-PEPS on the same small clip and matched budget')
elif TRACK == 'b':
    # int4/int8 and mixed/per-channel PTQ already exist; extend calibration or QAT.
    print('b) test clipping/calibration or short QAT at matched total encoded bits')
elif TRACK == 'c':
    # Add your aggregator to peps/aggregate.py + make_aggregator, then compare.
    print('c) compare your aggregator vs concat/pink/brownian at matched budget')
elif TRACK == 'd':
    # Optimize one full-pipeline component and preserve parity.
    print('d) compare one optimization against a rerun full-pipeline baseline')
else:
    raise ValueError(f'unknown TRACK {TRACK!r}')

## 3. Results table + figure -> CSV / 結果表 + 圖 -> CSV
Put baseline and your result side by side, write the CSV, and draw one figure.
把基線與你的結果並排,寫出 CSV,畫一張圖。

In [ ]:
import csv
os.makedirs('../results', exist_ok=True)
# TODO(student): set only after the extension run finishes.
EXTENSION_VALUE = None
EXTENSION_LABEL = None
if EXTENSION_VALUE is None or EXTENSION_LABEL is None:
    raise ValueError('Refusing to write a blank submission: fill extension value/label')
extension_value = float(EXTENSION_VALUE)
if not math.isfinite(extension_value): raise ValueError('extension value must be finite')
extension_label = _required_text(EXTENSION_LABEL, 'EXTENSION_LABEL')
metric = _required_text(baseline_row['metric'], 'baseline metric')
units = _required_text(baseline_row['units'], 'baseline units')
profile = baseline_row['profile']
seed = int(baseline_row['seed'])
rows = [
    ('baseline', metric, baseline_value, units, profile, seed, 'verified'),
    ('extension', metric, extension_value, units, profile, seed, 'verified'),
]
out = f'../results/capstone_{TRACK}_result.csv'
with open(out, 'w', newline='') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['stage', 'metric', 'value', 'units', 'profile', 'seed', 'status'])
    w.writerows(rows)
print('wrote', out)
plt.bar([row[0] for row in rows], [row[2] for row in rows])
plt.ylabel(f'{metric} ({units})')
plt.title(f'Capstone {TRACK}: matched baseline vs extension'); plt.show()

## 4. Make your slide / 做你的投影片
Drop a deck into `slides/` in the course's bilingual style (English title + 繁中
要點, `---` separators) and build it. Skeleton to copy:

以課程雙語風格(英文標題 + 繁中要點、`---` 分隔)在 `slides/` 放一份投影片並建置。
可複製的骨架:

In [ ]:
slide = '''---
marp: true
theme: default
paginate: true
title: Capstone — <your title>
---

# <Your project> / <你的專題>

- Baseline / 基線: <number>
- Change / 改動: <one sentence>
- Result / 結果: <number> (win or honest loss)
'''
# TODO(student): save as slides/06_capstone_<name>.md then build:
#   cd slides && make 06_capstone_<name>.pdf
print(slide)

## 5. Self-assessment vs rubric / 對照評分表自評
Score yourself against `docs/06_capstone.md`. Be honest — the rubric rewards a
matched baseline and clean reporting over a big-but-unfair number.

對照 `docs/06_capstone.md` 自評。誠實為上 —— 評分表看重相同條件的基線與乾淨呈現,
勝過又大又不公平的數字。

In [ ]:
rubric = {
  'baseline_correctness (30%)':     'baseline rerun; extension is correct',
  'extension_depth (25%)':          'the change is non-trivial and well-motivated',
  'empirical_rigor (25%)':          'matched params/bitrate/tolerance; fair compare',
  'honest_reporting (10%)':         'limitations + negative results stated plainly',
  'communication (10%)':            'notebook + CSV + manifests + bilingual slide',
}
# TODO(student): rate each 0-1 and justify in one line.
for k, v in rubric.items():
    print(f'[ ] {k}: {v}')

## 6. Reflection / honest limitations / 反思與誠實限制
Close like the course does: what worked, what didn't, and the one experiment you'd
run next. A clean negative result is a real contribution.

如課程般收尾:什麼有效、什麼無效、以及下一個你會做的實驗。一個乾淨的負結果也是
真正的貢獻。

## 7. Submission gate / 提交門檻
Copy `course/templates/capstone_submission.json`, fill every field, then run:

`python3 scripts/validate_submission.py <submission.json> --kind capstone`

The template intentionally fails while any placeholder, blank value, legacy
baseline, missing artifact, or non-finite CSV value remains.

複製 capstone submission 模板、填完全部欄位後執行 validator。任何 placeholder、
空值、legacy 基線、缺失 artifact 或非有限 CSV 值都會被拒絕。